In [43]:
!pip install requests beautifulsoup4 lxml -q

In [44]:
import requests
from bs4 import BeautifulSoup
import re

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

In [45]:
def scrape_filgoal_news(section_url="https://www.filgoal.com/section/122/في-المونديال"):


    response = requests.get(section_url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")

    news_list = []
    seen_links = set()

    for link_tag in soup.find_all("a", href=re.compile(r"/articles/\d+/")):
        link = link_tag.get("href")
        title = link_tag.get_text(strip=True)

        if not title or link in seen_links:
            continue

        if not link.startswith("http"):
            link = "https://www.filgoal.com" + link

        seen_links.add(link)
        news_list.append({"title": title, "link": link, "source": "FilGoal"})

    return news_list

In [46]:
filgoal_news = scrape_filgoal_news()
print(f"{len(filgoal_news)} news articles found in FilGoal's World Cup section.")

for news in filgoal_news[:5]:
    print(news["title"])
    print(news["link"])

35 news articles found in FilGoal's World Cup section.
كأس العالم – رودري أفضل لاعب في مباراة إسبانيا والبرتغال
https://www.filgoal.com/articles/532715/كأس-العالم-رودري-أفضل-لاعب-في-مباراة-إسبانيا-والبرتغال
كأس العالم - سكالوني: ميسي بخير.. ومصر ليست مثل كاب فيردي
https://www.filgoal.com/articles/532714/كأس-العالم-سكالوني-ميسي-بخير-ومصر-ليست-مثل-كاب-فيردي
عكس أمنية الدون
https://www.filgoal.com/articles/532713/عكس-أمنية-الدون
كأس العالم - مؤتمر حسام حسن: سنفرض شخصيتنا على الأرجنتين.. وأستكمل حاليا طموحاتي كلاعب
https://www.filgoal.com/articles/532712/كأس-العالم-مؤتمر-حسام-حسن-سنفرض-شخصيتنا-على-الأرجنتين-وأستكمل-حاليا-طموحاتي-كلاعب
كأس العالم – حسام حسن: من يؤذي كلبا يعاقب فماذا عن من يقتلون الأبرياء في فلسطين
https://www.filgoal.com/articles/532711/كأس-العالم-حسام-حسن-من-يؤذي-كلبا-يعاقب-فماذا-عن-من-يقتلون-الأبرياء-في-فلسطين


In [47]:
def scrape_filgoal_matches(matches_url="https://www.filgoal.com/matches"):

    response = requests.get(matches_url, headers=HEADERS, timeout=15)
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "lxml")

    team_links = soup.find_all("a", href=re.compile(r"/teams/\d+/"))

    matches = []
    seen_match_ids = set()

    for i in range(0, len(team_links) - 1, 2):
        home_link = team_links[i]
        away_link = team_links[i + 1]

        home_name = home_link.get_text(strip=True)
        away_name = away_link.get_text(strip=True)

        if not home_name or not away_name:
            continue

        container = home_link
        match_link_tag = None
        for _ in range(6):
            if container.parent is None:
                break
            container = container.parent
            found = container.find("a", href=re.compile(r"/matches/\d+/"))
            if found:
                match_link_tag = found
                break

        if match_link_tag is None:
            continue

        match_href = match_link_tag.get("href")
        match_id_search = re.search(r"/matches/(\d+)/", match_href)
        if not match_id_search:
            continue
        match_id = match_id_search.group(1)

        if match_id in seen_match_ids:
            continue
        seen_match_ids.add(match_id)

        full_text = container.get_text(" ", strip=True)

        if "انتهت" in full_text:
            status = "انتهت"
        elif "مباشر" in full_text:
            status = "مباشر"
        else:
            status = "لم تبدأ"

        date_match = re.search(r"(\d{2}-\d{2}-\d{4})\s*-\s*(\d{2}:\d{2})", full_text)
        match_date = date_match.group(1) if date_match else None
        match_time = date_match.group(2) if date_match else None

        link = match_href if match_href.startswith("http") else "https://www.filgoal.com" + match_href

        matches.append({
            "match_id": match_id,
            "home_team": home_name,
            "away_team": away_name,
            "status": status,
            "date": match_date,
            "time": match_time,
            "link": link,
            "source": "FilGoal",
        })

    return matches

In [48]:
filgoal_matches = scrape_filgoal_matches()
print(f"{len(filgoal_matches)} matches found in FilGoal's matches section.")

for m in filgoal_matches[:5]:
    print(f"{m['home_team']} vs {m['away_team']} - {m['status']}")

3 matches found in FilGoal's matches section.
سويسرا vs كولومبيا - لم تبدأ
الأرجنتين vs مصر - لم تبدأ
أمريكا vs بلجيكا - لم تبدأ


In [49]:
WORLD_CUP_KEYWORD = "أمريكا-2026"

world_cup_matches = [m for m in filgoal_matches if WORLD_CUP_KEYWORD in m["link"]]
print(f"{len(world_cup_matches)} matches found in FilGoal's World Cup section.")

for m in world_cup_matches[:5]:
    print(f"{m['home_team']} vs {m['away_team']} - {m['status']}")

3 matches found in FilGoal's World Cup section.
سويسرا vs كولومبيا - لم تبدأ
الأرجنتين vs مصر - لم تبدأ
أمريكا vs بلجيكا - لم تبدأ


In [50]:
with open("filgoal_news.txt", "w", encoding="utf-8") as f:
    for news in filgoal_news:
        f.write(news["title"] + "\n")
        f.write(news["link"] + "\n")

print("filgoal_news.txt")

filgoal_news.txt


In [51]:
with open("filgoal_matches.txt", "w", encoding="utf-8") as f:
    for m in filgoal_matches:
        f.write(f"{m['home_team']} vs {m['away_team']}\n")
        f.write(f"Status: {m['status']}\n")
        if m["date"]:
            f.write(f"Date: {m['date']} - {m['time']}\n")
        f.write(f"Link: {m['link']}\n")

print("filgoal_matches.txt")

filgoal_matches.txt


In [52]:
with open("world_cup_matches.txt", "w", encoding="utf-8") as f:
    for m in world_cup_matches:
        f.write(f"{m['home_team']} vs {m['away_team']}\n")
        f.write(f"Status: {m['status']}\n")
        if m["date"]:
            f.write(f"Date: {m['date']} - {m['time']}\n")
        f.write(f"Link: {m['link']}\n")

print("world_cup_matches.txt")
print("\nAll files are ready")

world_cup_matches.txt

All files are ready


In [53]:
def get_wikipedia_text(title):

    url = "https://en.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",   
        "explaintext": 1,     
        "titles": title,
        "format": "json",
    }

    response = requests.get(url, headers=HEADERS, params=params, timeout=15)
    response.raise_for_status()

    data = response.json()
    pages = data["query"]["pages"]
    page = list(pages.values())[0]

    return page.get("extract", "")

In [54]:
worldcup_history_text = get_wikipedia_text("FIFA_World_Cup")
print(f"Number of characters: {len(worldcup_history_text)}")
print(worldcup_history_text[:500])

Number of characters: 36464
The FIFA World Cup is an international association football competition among the senior men's national teams of the members of the Fédération Internationale de Football Association (FIFA), the sport's global governing body. The tournament has been held every four years since the inaugural tournament in 1930, with the exception of 1942 and 1946 due to World War II. The reigning champions are Argentina, who won their third title at the 2022 World Cup by defeating France.
The contest starts with t


In [55]:
current_wc_text = get_wikipedia_text("2026_FIFA_World_Cup")
print(f"Number of characters: {len(current_wc_text)}")
print(current_wc_text[:500])

Number of characters: 41742
The 2026 FIFA World Cup is the 23rd and current FIFA World Cup, the quadrennial international men's soccer championship contested by the national teams of the member associations of FIFA. The tournament began on June 11, 2026, and will conclude on July 19. It is jointly hosted by 16 cities—11 in the United States, 3 in Mexico, and 2 in Canada. The tournament is the first FIFA World Cup to be hosted by three nations and the first to include 48 teams, an expansion from the previous 32-team format.


In [56]:
with open("worldcup_history_text.txt", "w", encoding="utf-8") as f:
    f.write(worldcup_history_text)

print("worldcup_history_text.txt")

with open("current_wc_text.txt", "w", encoding="utf-8") as f:
    f.write(current_wc_text)

print("current_wc_text.txt")

worldcup_history_text.txt
current_wc_text.txt
